# Summary Memory

> **Compress older turns with an LLM-generated rolling summary: trade word-for-word recall for unbounded conversation length.**

In the [previous notebook](../02_sliding_window_memory/sliding_window_memory.ipynb) we saw that **Sliding Window Memory** bounds cost by discarding old messages entirely. That hard cutoff means the agent doesn't even *know* it forgot something.

**Summary Memory** takes a different approach. Instead of discarding history, it *compresses* it. Think of it like reading a long book and writing a one-page summary at the end of each chapter. You can't quote the book word-for-word anymore, but you still know what happened. A secondary LLM call periodically condenses older messages into a running textual summary. The agent loses exact wording but retains the gist (key facts, decisions, and context) across arbitrarily long conversations.

**The catch:** summaries are lossy (they can't perfectly reconstruct the original). Each compression cycle can lose details, shift emphasis, or subtly distort facts. Over many cycles this **summary drift** compounds. The agent's "memory" can diverge from what actually happened.

**By the end of this notebook you'll understand:**
- How to build a rolling-summary memory system from scratch with the Anthropic SDK.
- The summarizer loop: when to trigger it, what prompt to use, and how the summary evolves.
- How summaries drift over long conversations, with a controlled experiment and visualizations.
- Practical mitigations for drift and information loss.

## Key Concepts

- **Rolling summary**: A single text block that gets incrementally updated as the conversation grows. Each update folds new messages into the existing summary.
- **Summarizer prompt**: The instruction given to the LLM to produce the summary. Its wording controls what's preserved (facts, decisions, tone) and what's discarded.
- **Refresh trigger**: The rule that decides *when* to re-summarize. Common options: after every *n* messages, when the buffer exceeds a token threshold, or on every turn.
- **Summary drift**: The gradual distortion of facts across repeated summarization cycles. Details get softened, merged, or lost entirely over time.
- **Compression ratio**: How much shorter the summary is compared to the raw messages it replaces. Higher compression means more information loss.
- **Buffer zone**: Recent messages kept word-for-word alongside the summary. This gives the LLM exact context for the latest exchanges while older context lives in the compressed summary.

## Architecture

<p align="center">
  <img src="../../images/diagrams/03_summary_memory.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
sequenceDiagram
    participant U as User
    participant B as Message Buffer
    participant S as Summary Store
    participant LLM as Claude (Chat)
    participant SLLM as Claude (Summarizer)

    Note over S: Summary: "" (empty)
    Note over B: Buffer: []

    U->>B: msg 1
    B->>LLM: [summary="", msg1]
    LLM-->>B: msg 2
    Note over B: Buffer: [msg1, msg2]

    U->>B: msg 3
    B->>LLM: [summary="", msg1..msg3]
    LLM-->>B: msg 4
    Note over B: Buffer: [msg1..msg4] - trigger!

    rect rgb(255, 245, 230)
        Note over B,SLLM: 🔄 Summarization Cycle
        B->>SLLM: "Summarize: {old_summary} + {msg1..msg4}"
        SLLM-->>S: Updated summary
        B->>B: Clear buffer
    end

    U->>B: msg 5
    B->>LLM: [summary="...", msg5]
    LLM-->>B: msg 6
    Note over B: Buffer: [msg5, msg6]
    Note over S: Summary carries forward<br/>compressed history
```

</details>

In [ ]:
# Install required packages (run once)
%pip install -q anthropic python-dotenv matplotlib pandas numpy

## Setup

Import the Anthropic SDK and load your API key from the `.env` file.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads ANTHROPIC_API_KEY from .env

import anthropic

assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your .env file"
print("✓ Anthropic API key loaded")

## Core Implementation

The design has two parts:

1. **Buffer**: recent messages stored word-for-word (a list of dicts).
2. **Summary**: a string that compresses all older messages.

When the buffer reaches `max_buffer_size` messages, we call the LLM with a *summarizer prompt*. That prompt folds the current summary + buffer into a new summary. Then we clear the buffer.

On each chat turn the LLM receives: `[system: summary_context] + buffer_messages`.

In [ ]:
SUMMARIZER_PROMPT = """You are a conversation summarizer. Given the existing summary and new messages,
produce an updated summary that:
1. Preserves all key FACTS (names, numbers, preferences, decisions).
2. Notes any open questions or unresolved topics.
3. Stays concise (aim for 2-5 sentences).

Current summary:
{summary}

New messages:
{messages}

Write only the updated summary, nothing else."""




Now we define the `SummaryMemory` class. The constructor sets up two storage areas: a `summary` string (initially empty) and a `buffer` list for recent messages. The `_summarize` method calls the LLM to compress the current summary plus buffer into a fresh, shorter summary.

In [ ]:
class SummaryMemory:
    """Rolling-summary memory that compresses older turns via an LLM."""

    def __init__(
        self,
        max_buffer_size: int = 6,
        model: str = "claude-sonnet-4-20250514",
        summarizer_model: str = "claude-sonnet-4-20250514",
        system_prompt: str | None = None,
        max_tokens: int = 1024,
    ):
        self.client = anthropic.Anthropic()
        self.model = model
        self.summarizer_model = summarizer_model
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self.max_buffer_size = max_buffer_size

        # State
        self.summary: str = ""
        self.buffer: list[dict] = []

        # Tracking
        self.full_history: list[dict] = []
        self.summary_history: list[str] = []  # every summary version
        self.turn_token_usage: list[dict] = []
        self._summary_calls = 0

    # ── Summarization ────────────────────────────────────────────
    def _format_messages_for_summary(self, messages: list[dict]) -> str:
        lines = []
        for msg in messages:
            role = "User" if msg["role"] == "user" else "Assistant"
            lines.append(f"{role}: {msg['content']}")
        return "\n".join(lines)

    def _summarize(self) -> str:
        """Compress current summary + buffer into a new summary."""
        messages_text = self._format_messages_for_summary(self.buffer)
        prompt = SUMMARIZER_PROMPT.format(
            summary=self.summary or "(no prior summary)",
            messages=messages_text,
        )
        response = self.client.messages.create(
            model=self.summarizer_model,
            max_tokens=512,
            messages=[{"role": "user", "content": prompt}],
        )
        self._summary_calls += 1
        return response.content[0].text.strip()



The `chat` method is where the magic happens. It builds a system prompt that includes the running summary, sends the buffer to the LLM, and checks whether the buffer has grown past `max_buffer_size`. If so, it triggers a summarization cycle and clears the buffer.

In [ ]:
    # ── Chat ─────────────────────────────────────────────────────
    def chat(self, user_input: str) -> str:
        user_msg = {"role": "user", "content": user_input}
        self.buffer.append(user_msg)
        self.full_history.append(user_msg)

        # Build the system prompt with summary context
        system_parts = []
        if self.system_prompt:
            system_parts.append(self.system_prompt)
        if self.summary:
            system_parts.append(f"Conversation summary so far:\n{self.summary}")
        system = "\n\n".join(system_parts) if system_parts else None

        kwargs = dict(
            model=self.model,
            max_tokens=self.max_tokens,
            messages=self.buffer,
        )
        if system:
            kwargs["system"] = system

        response = self.client.messages.create(**kwargs)
        assistant_text = response.content[0].text

        assistant_msg = {"role": "assistant", "content": assistant_text}
        self.buffer.append(assistant_msg)
        self.full_history.append(assistant_msg)

        self.turn_token_usage.append({
            "turn": len(self.turn_token_usage) + 1,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "buffer_msgs": len(self.buffer),
            "summary_len": len(self.summary),
        })

        # Check if we need to summarize
        if len(self.buffer) >= self.max_buffer_size:
            self.summary = self._summarize()
            self.summary_history.append(self.summary)
            self.buffer.clear()

        return assistant_text



Finally, we add inspection and utility methods. `get_context_snapshot` shows you exactly what the LLM would see on the next call: the summary text plus the current buffer contents.

In [ ]:
    # ── Inspection ───────────────────────────────────────────────
    def get_context_snapshot(self) -> dict:
        """Return what the LLM currently sees."""
        return {
            "summary": self.summary,
            "buffer": list(self.buffer),
            "buffer_size": len(self.buffer),
            "total_messages": len(self.full_history),
            "summary_versions": len(self.summary_history),
        }

    def clear(self) -> None:
        self.summary = ""
        self.buffer.clear()
        self.full_history.clear()
        self.summary_history.clear()
        self.turn_token_usage.clear()
        self._summary_calls = 0

    def __repr__(self) -> str:
        return (
            f"SummaryMemory(buffer={len(self.buffer)}/{self.max_buffer_size}, "
            f"summary_versions={len(self.summary_history)}, "
            f"total_msgs={len(self.full_history)})"
        )


print("✓ SummaryMemory class defined")

## Usage Example: Watching the Summary Evolve

Let's use a small buffer (`max_buffer_size=4`, meaning 2 turns trigger a summarization). We'll plant several facts and watch how they get compressed into the summary.

In [ ]:
mem = SummaryMemory(
    max_buffer_size=4,  # summarize every 2 turns
    system_prompt="You are a concise assistant. Reply in 1-2 sentences.",
)

conversation = [
    "My name is Carlos and I'm from Buenos Aires.",
    "I work as a marine biologist studying coral reefs.",
    "My favorite programming language is Rust.",
    "What do you know about me so far?",
]

for msg in conversation:
    print(f"👤 User:  {msg}")
    reply = mem.chat(msg)
    print(f"🤖 Agent: {reply}")
    snap = mem.get_context_snapshot()
    print(f"   📊 Buffer: {snap['buffer_size']}/{mem.max_buffer_size} | Summary versions: {snap['summary_versions']}")
    if snap["summary"]:
        print(f"   📝 Current summary: {snap['summary'][:120]}...")
    print()

Let's trace how the summary evolved over time. Each version shows what the summarizer produced after folding in a batch of messages. We also print the current buffer contents.

In [ ]:
print("=== Summary Evolution ===\n")
for i, s in enumerate(mem.summary_history):
    print(f"Version {i+1}:")
    print(f"  {s}")
    print()

print(f"=== Current Buffer ({len(mem.buffer)} messages) ===")
for msg in mem.buffer:
    role = "USER" if msg["role"] == "user" else "ASST"
    print(f"  {role}: {msg['content'][:80]}")

## Experiment: Summary Drift Over Long Conversations

Summary drift is the most important failure mode of this technique. Each summarization cycle is lossy, and errors compound. A fact stated clearly in turn 1 might be softened in summary v2, vaguely referenced in v3, and gone entirely by v5.

Let's run a controlled experiment:
1. Plant **5 specific facts** in the first 5 turns.
2. Continue for **15 more filler turns** (triggering multiple re-summarizations).
3. After each summarization cycle, check whether each original fact still appears in the summary.
4. Visualize how fact retention degrades over summary versions.

In [ ]:
import re

FACTS = [
    ("My full name is Elena Vasquez.", "elena vasquez"),
    ("I was born on March 15, 1992.", "1992"),
    ("I have exactly three cats named Mochi, Tofu, and Tempeh.", "tempeh"),
    ("My salary is $145,000 per year.", "145"),
    ("I'm allergic to strawberries.", "strawberr"),
]

FILLER_MESSAGES = [
    "What's a good recipe for pasta carbonara?",
    "Tell me about the history of jazz music.",
    "How do black holes form?",
    "What are the main differences between Python and JavaScript?",
    "Can you explain quantum entanglement simply?",
    "What's the tallest building in the world?",
    "How does a combustion engine work?",
    "What are some tips for better sleep?",
    "Tell me about the migration patterns of monarch butterflies.",
    "What causes the northern lights?",
    "How do vaccines work?",
    "What's the deepest point in the ocean?",
    "Tell me about the history of chess.",
    "How does GPS work?",
    "What are prime numbers used for in cryptography?",
]

# Run experiment with a small buffer to force many summarization cycles
mem_drift = SummaryMemory(
    max_buffer_size=4,  # summarize every 2 turns
    system_prompt="You are a helpful assistant. Reply concisely in 1-2 sentences.",
)

# Phase 1: Plant facts
print("Phase 1: Planting facts...")
for fact_text, _ in FACTS:
    mem_drift.chat(fact_text)
print(f"  Planted {len(FACTS)} facts. Summary versions so far: {len(mem_drift.summary_history)}")

# Phase 2: Filler to cause repeated re-summarization
print("\nPhase 2: Filler turns (causing re-summarization)...")
for i, filler in enumerate(FILLER_MESSAGES):
    mem_drift.chat(filler)

print(f"  Completed {len(FILLER_MESSAGES)} filler turns.")
print(f"  Total summary versions: {len(mem_drift.summary_history)}")
print(f"  Total LLM summarization calls: {mem_drift._summary_calls}")

Now we check which facts survived in each summary version. We search for keywords (like "elena vasquez" or "1992") in each summary. A checkmark means the fact is present. A cross means it was lost during compression.

In [ ]:
# Check which facts survive in each summary version
fact_labels = [f[0][:35] + "..." for f in FACTS]
fact_keywords = [f[1] for f in FACTS]

retention_matrix = []
for summary in mem_drift.summary_history:
    row = []
    for keyword in fact_keywords:
        present = keyword.lower() in summary.lower()
        row.append(1 if present else 0)
    retention_matrix.append(row)

# Print the retention table
print("Fact Retention Across Summary Versions")
print("=" * 70)
header = f"{'Version':<10}" + "".join(f"{label:<12}" for label in ["Elena V.", "Born 1992", "3 cats", "$145K", "Strawberry"])
print(header)
print("-" * 70)
for i, row in enumerate(retention_matrix):
    cells_str = "".join(f"{'  ✓':<12}" if v else f"{'  ✗':<12}" for v in row)
    print(f"v{i+1:<9}{cells_str}")

total_facts = len(FACTS)
final_retained = sum(retention_matrix[-1]) if retention_matrix else 0
print(f"\nFinal retention: {final_retained}/{total_facts} facts survived to the last summary version.")

Visualize fact retention across summary versions. The heatmap (left) shows which facts survive in each version. The bar chart (right) shows the total count of retained facts. Watch how the count drops as summarization cycles accumulate.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

versions = list(range(1, len(retention_matrix) + 1))
fact_names = ["Elena V.", "Born 1992", "3 Cats", "$145K Salary", "Strawberry Allergy"]

# Left: Heatmap of fact retention
retention_arr = np.array(retention_matrix)
im = ax1.imshow(retention_arr.T, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
ax1.set_xticks(range(len(versions)))
ax1.set_xticklabels([f"v{v}" for v in versions], fontsize=8)
ax1.set_yticks(range(len(fact_names)))
ax1.set_yticklabels(fact_names)
ax1.set_xlabel("Summary Version")
ax1.set_title("Fact Retention Heatmap\n(Green = present, Red = lost)")

# Add text annotations
for i in range(len(versions)):
    for j in range(len(fact_names)):
        symbol = "✓" if retention_arr[i, j] else "✗"
        ax1.text(i, j, symbol, ha="center", va="center",
                 color="white" if retention_arr[i, j] == 0 else "black", fontsize=10)

# Right: Total facts retained per version
totals = [sum(row) for row in retention_matrix]
colors = ["#22c55e" if t >= 4 else "#f59e0b" if t >= 2 else "#ef4444" for t in totals]
ax2.bar(range(len(versions)), totals, color=colors, alpha=0.85)
ax2.set_xticks(range(len(versions)))
ax2.set_xticklabels([f"v{v}" for v in versions])
ax2.set_ylabel("Facts Retained")
ax2.set_ylim(0, len(FACTS) + 0.5)
ax2.set_xlabel("Summary Version")
ax2.set_title("Total Facts Retained per Summary Version")
ax2.axhline(y=len(FACTS), color="gray", linestyle="--", alpha=0.3, label=f"All {len(FACTS)} facts")
ax2.legend()

for i, t in enumerate(totals):
    ax2.text(i, t + 0.15, str(t), ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("summary_drift.png", dpi=150, bbox_inches="tight")
plt.show()

print("Summary drift is visible: facts degrade as summarization cycles accumulate.")

Print the full text of each summary version. Reading them in sequence, you can see how the language shifts and details drop out. This is summary drift in action.

In [ ]:
print("=== Full Summary Evolution ===\n")
for i, s in enumerate(mem_drift.summary_history):
    retained = sum(retention_matrix[i])
    print(f"--- Version {i+1} ({retained}/{len(FACTS)} facts retained) ---")
    print(s)
    print()

## Token Cost: Summary Memory vs. Full Buffer vs. Sliding Window

Summary memory sits between full buffer (unlimited recall, quadratic cost) and sliding window (hard cutoff, constant cost). Let's compare all three approaches.

In [ ]:
import matplotlib.pyplot as plt

# Simulate token growth for 30 turns
# Assumptions: ~50 tokens/message, ~30 tokens system prompt, summary ~100 tokens
MSG_TOKENS = 50
SYS_TOKENS = 30
SUMMARY_TOKENS = 100
NUM_TURNS = 30
WINDOW_K = 10
BUFFER_SIZE = 6  # summarize every 3 turns

buffer_cost, window_cost, summary_cost = [], [], []

for turn in range(1, NUM_TURNS + 1):
    n_msgs = turn * 2

    # Full buffer: all messages
    buffer_cost.append(SYS_TOKENS + n_msgs * MSG_TOKENS)

    # Sliding window: capped at k
    window_cost.append(SYS_TOKENS + min(n_msgs, WINDOW_K) * MSG_TOKENS)

    # Summary memory: summary + buffer (buffer resets periodically)
    buffer_msgs = (n_msgs % BUFFER_SIZE) or BUFFER_SIZE
    summary_cost.append(SYS_TOKENS + SUMMARY_TOKENS + buffer_msgs * MSG_TOKENS)

turns = list(range(1, NUM_TURNS + 1))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(turns, buffer_cost, "o-", color="#ef4444", label="Full Buffer", linewidth=2, markersize=4)
ax.plot(turns, window_cost, "s-", color="#22c55e", label=f"Sliding Window (k={WINDOW_K})", linewidth=2, markersize=4)
ax.plot(turns, summary_cost, "^-", color="#6366f1", label=f"Summary Memory (buf={BUFFER_SIZE})", linewidth=2, markersize=4)

ax.set_xlabel("Conversation Turn")
ax.set_ylabel("Input Tokens per API Call")
ax.set_title("Input Tokens per Turn: Three Memory Strategies")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("cost_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"At turn {NUM_TURNS}:")
print(f"  Full buffer:    {buffer_cost[-1]:,} tokens")
print(f"  Sliding window: {window_cost[-1]:,} tokens")
print(f"  Summary memory: {summary_cost[-1]:,} tokens")
print(f"\nNote: Summary memory also incurs extra cost for summarization calls (not shown).")

## Recall Test: Can the Agent Still Use Its Summary?

Let's verify that the agent can actually use facts from its summary to answer questions. Remember: it no longer has the original messages.

In [ ]:
RECALL_QUESTIONS = [
    ("What is my full name?", "elena"),
    ("When was I born?", "1992"),
    ("What are my cats' names?", "mochi"),
    ("What is my salary?", "145"),
    ("What food am I allergic to?", "strawberr"),
]

print("=== Recall Test (after 20 total turns) ===\n")
results = []
for question, keyword in RECALL_QUESTIONS:
    answer = mem_drift.chat(question)
    recalled = keyword.lower() in answer.lower()
    status = "✓ RECALLED" if recalled else "✗ FORGOTTEN"
    results.append(recalled)
    print(f"  Q: {question}")
    print(f"  A: {answer[:120]}")
    print(f"  {status}")
    print()

score = sum(results)
print(f"Score: {score}/{len(RECALL_QUESTIONS)} facts recalled from summary.")

## Mitigating Summary Drift

Summary drift is inherent to this technique, but several strategies reduce its impact:

### 1. Better Summarizer Prompts
Be explicit about what to preserve:
```
Preserve ALL of the following from the conversation:
- Proper nouns (names, places, organizations)
- Numbers (dates, amounts, quantities)
- Stated preferences and constraints
- Decisions made and their rationale
```

### 2. Structured Summaries
Instead of free-form text, use a structured format:
```
FACTS: [name: Elena Vasquez, born: 1992-03-15, ...]
DECISIONS: [chose Python over Java, ...]
OPEN QUESTIONS: [budget not yet decided, ...]
```
This makes it harder for the LLM to accidentally drop fields.

### 3. Larger Buffer Sizes
Fewer summarization cycles means less drift. If your context window allows it, use `max_buffer_size=20` instead of `max_buffer_size=4`.

### 4. Summary + Buffer Hybrid
Keep the last *k* messages word-for-word alongside the summary. This is the approach used in production systems like LangChain's `ConversationSummaryBufferMemory`. We cover it in [technique 04](../04_summary_buffer_memory/).

### 5. Entity Extraction
Extract key entities into a separate store and inject them alongside the summary. Even if the summary drifts, the entity store preserves structured facts.

### 6. Periodic Full Re-summarization
Instead of always folding incrementally, occasionally re-summarize from a larger chunk of raw history. This reduces compounding errors from chain-of-summaries.

## Discussion and Tradeoffs

### Strengths
- **Unbounded conversations**: Unlike sliding window, summary memory can handle arbitrarily long conversations without losing all older context.
- **Bounded token cost**: The summary is compact. Per-turn cost is roughly constant (summary + buffer).
- **Graceful degradation**: Instead of a hard cutoff, information compresses gradually. Important themes tend to survive longer than minor details.
- **Flexible compression**: The summarizer prompt controls what's preserved. You can tune it for your domain.

### Weaknesses
- **Summary drift**: Repeated compression introduces cumulative information loss. Specific numbers, names, and details are most vulnerable.
- **Extra LLM calls**: Each summarization cycle costs tokens and adds latency (the delay before a response arrives). With an aggressive buffer size of 4, you're making an extra LLM call every 2 turns.
- **Non-deterministic memory**: Two runs of the same conversation may produce different summaries. This leads to different agent behavior.
- **Hard to debug**: When the agent "forgets" something, it's hard to tell whether the summarizer dropped it or the chat model ignored it.
- **No exact recall**: The agent can never quote you word-for-word from summarized history.

### When to Use Summary Memory

| Scenario | Recommendation |
|----------|---------------|
| Long conversations (50+ turns) | Good fit. Bounds cost while retaining the gist. |
| Need exact recall of early facts | Not ideal. Use buffer or retrieval-augmented memory instead. |
| Cost-sensitive applications | Good fit. Much cheaper than full buffer. |
| Short conversations (under 10 turns) | Overhead not worth it. Use buffer or window. |
| Multi-session agents | Good fit. A summary carries over between sessions easily. |

### Cost Model
For *n* turns with buffer size *b*:
- **Chat calls:** *n* (same as any approach)
- **Summary calls:** floor(*n* / (*b*/2)) (one per buffer flush)
- **Total extra cost:** proportional to *n/b*. That's a small fraction of total cost when *b* is reasonable.

## Further Reading

- [LangChain ConversationSummaryMemory](https://python.langchain.com/docs/modules/memory/types/summary?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Production implementation with customizable prompts
- [LangChain ConversationSummaryBufferMemory](https://python.langchain.com/docs/modules/memory/types/summary_buffer?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Hybrid approach combining summary with a token-limited buffer
- [Anthropic: Building Conversational AI](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Official multi-turn conversation patterns
- [Recursive Summarization with LLMs](https://arxiv.org/abs/2301.13848): Research on iterative summarization quality
- [Lilian Weng, "LLM Powered Autonomous Agents"](https://lilianweng.github.io/posts/2023-06-23-agent/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Overview including memory architectures

---

*← Previous: 02: Sliding Window Memory · Next: [04: Summary Buffer Memory](../04_summary_buffer_memory/) →*

In [ ]:
# Clean up temp files created during the demo
import os
for f in ["summary_drift.png", "cost_comparison.png"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: Prompt engineering for summaries
Write three different `SUMMARIZER_PROMPT` variants: one that prioritizes facts, one that prioritizes user preferences, and one that prioritizes action items. Run the same 10-turn conversation with each and compare the resulting summaries side by side.

### Challenge 2: Quantify summary drift
Extend the drift experiment to 30 turns. After each summarization pass, use `extract_topic_tags()` to capture topics. Plot the number of surviving original-turn topics over time. Calculate the half-life of a topic in the summary.

### Challenge 3: Two-tier summary
Maintain two summaries: a detailed one (last 10 turns) and a high-level one (everything before that). When the detailed summary grows past a threshold, compress it into the high-level summary. This mirrors the pattern in 15 Memory Compaction.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--03-summary-memory--summary-memory)
